# Experimenting with effect of missing uv-coverage or noise in SA data
## A. Ordog, Feb 13, 2023
### Feb 14, 2023: added simple gap test
### Feb 21, 2023: option to test different feathering and optimize
### Feb 22, 2023: option to run multiple simulations and optimize feathering for each
### Oct 08, 2024: correct uv coordinate axis and make more functions for repeated procedures

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
from matplotlib import pylab
from PIL import Image
from math import e
import astropy.io.fits as pf
from astropy.io import fits
from mpl_toolkits.axes_grid1 import make_axes_locatable
from tqdm import tqdm
from astropy.convolution import convolve
from astropy.convolution import Gaussian2DKernel

## Read in files (change directory names accordingly)

In [ ]:
#######################################################
#dir_in = '/home/aordog/DATA/CGPS_FITS_files/' # on faraday
#dir_in = '/home2/DATA_AO/CGPS_FITS_files/'    # on DRAO desktop
dir_in = '/srv/data/cgps-gmims/cgps_for_sims/' # on elephant
#######################################################

CGPS = fits.open(dir_in+'CGPS_C21.fits')
hdr = CGPS[0].header
data_full = CGPS[0].data[0,0]
print(data_full.shape)
print(hdr['CDELT1'])
#print(repr(hdr))
#print('')

CGPS2 = fits.open(dir_in+'all_stokesi_allbands9.fits')
hdr2 = CGPS2[0].header
data_ST = CGPS2[0].data#[0,0]
print(data_ST.shape)
print(hdr2['CDELT1'])
#print(repr(hdr2))

## Define functions

### Basic functions

In [ ]:
def gf(mu,fwhm,x):
    return np.exp(-4*np.log(2)*((x-mu)**2)/fwhm**2)

In [ ]:
#def make_gaussian_lp(mu,fwhm,x):

#    gauss = np.exp(-4*np.log(2)*((x-mu)**2)/fwhm**2)

#    b = np.pi/(R2-R1)
#    r0 = (R1+R2)/2.

        
#    print('Applying high pass filter')
#    mask = 0.5*np.sin(b*(ruv - r0))+0.5
#    mask[ruv > R2] = 1
#    mask[ruv < R1] = 0



#    return mask

In [ ]:
def convolve_to_beam(map, biggest_beam, current_beam,dxy, output = False):
    
    fwhm = np.sqrt(biggest_beam**2 - current_beam**2) * 60

    if output: print('Convolving', np.round(current_beam,2), 'deg to', np.round(biggest_beam,2), 'deg\t FWHM of kernel: '+str(fwhm/60)+' degrees')   
    pixsize = dxy*60
    
    if fwhm < pixsize:
        if output: print("Just returning the map without change")
        return map
    
    if output: print('pixel size: '+str(pixsize)+' arcmin')   
    stddev = np.round((fwhm/2.355)/pixsize,0)
    if output: print('standard deviation of kernel: '+str(stddev)+' pixels')
    
    if stddev == 0:
        return map

    kernel = Gaussian2DKernel(x_stddev=stddev,y_stddev=stddev)
    
    return convolve(map,kernel), kernel

In [ ]:
def get_uv_axis(image, dxy, freqMHz=1420, method='right'):

    uv_freq_m = np.fft.fftshift(np.fft.fftfreq(image.shape[1]))/dxy
    uv_freq_n = np.fft.fftshift(np.fft.fftfreq(image.shape[0]))/dxy

    if method == 'right':
        #uv_m = 2*uv_freq_m*(3e8/(freqMHz*1e6))*180/np.pi # corrected Oct 2024
        #uv_n = 2*uv_freq_n*(3e8/(freqMHz*1e6))*180/np.pi # corrected Oct 2024
        uv_m = uv_freq_m*(3e8/(freqMHz*1e6))*180/np.pi # corrected Oct 2024
        uv_n = uv_freq_n*(3e8/(freqMHz*1e6))*180/np.pi # corrected Oct 2024
    if method == 'wrong':
        uv_m = uv_freq_m*(3e8/(freqMHz*1e6))*180/(np.pi**2)
        uv_n = uv_freq_n*(3e8/(freqMHz*1e6))*180/(np.pi**2)

    xuv, yuv = np.meshgrid(uv_m, uv_m)
    ruv = np.sqrt(xuv**2+yuv**2)

    uv = {}
    uv['uvm'] = uv_m
    uv['uvn'] = uv_n
    uv['xuv'] = xuv
    uv['yuv'] = yuv
    uv['ruv'] = ruv
    
    return uv

In [ ]:
def get_uvbounds_idx(uv,maxu,maxv):

    du = uv['uvm'][1] - uv['uvm'][0]
    dv = uv['uvn'][1] - uv['uvn'][0]

    print(du,dv)

    idx_u1 = abs(uv['uvm'] + maxu).argmin()
    idx_u2 = abs(uv['uvm'] - maxu).argmin()
    idx_v1 = abs(uv['uvn'] + maxv).argmin()
    idx_v2 = abs(uv['uvn'] - maxv).argmin()

    print(uv['uvm'][idx_u1],uv['uvm'][idx_u2])
    print(uv['uvn'][idx_v1],uv['uvn'][idx_v2])

    idx = {}
    idx['u1'] = idx_u1
    idx['u2'] = idx_u2
    idx['v1'] = idx_v1
    idx['v2'] = idx_v2

    extent = [uv['uvm'][idx_u1]-du/2, uv['uvm'][idx_u2]+du/2,
              uv['uvn'][idx_v1]-dv/2, uv['uvn'][idx_v2]+dv/2]
    
    return idx, extent

In [ ]:
def make_uv_mask(filter, R1, R2, ruv):

    b = np.pi/(R2-R1)
    r0 = (R1+R2)/2.

    if filter == 'hp':
        
        print('Applying high pass filter')
        mask = 0.5*np.sin(b*(ruv - r0))+0.5
        mask[ruv >= R2] = 1
        mask[ruv <= R1] = 0

    if filter == 'lp':
        
        print('Applying low pass filter')
        mask = -0.5*np.sin(b*(ruv - r0))+0.5
        mask[ruv >= R2] = 0
        mask[ruv <= R1] = 1

    return mask

In [ ]:
def regular_image(data,j0,i0,nxy,taper=False,taper_width=50,*args,**kwargs):

    image = data[j0:j0+nxy,i0:i0+nxy]

    dxy = np.round(hdr['CDELT2'],5)
    pix0 = int((nxy-1)/2)
    widxy = nxy*dxy
    
    if taper:      
        x, y = np.meshgrid(np.linspace(-(nxy-1)/2,(nxy-1)/2,nxy), 
                       np.linspace(-(nxy-1)/2,(nxy-1)/2,nxy))
        r = np.sqrt(x**2+y**2)
        
        taper = gf(nxy/2-taper_width,taper_width,r)
        taper[np.where(r<=nxy/2-taper_width)] = 1
        image = image*taper

    print('Width of each pixel: '+str(dxy)+ ' deg.')
    print('Number of x and y pixels: '+str(nxy))
    print('Central pixel index: '+str(pix0))
    print('Width of image: '+str(widxy)+' deg.')
    
    return image,dxy,pix0,widxy

In [ ]:
def padded_image(data,j0,i0,nxy,taper=False,taper_width=50,*args,**kwargs):

    image_small = data[j0:j0+nxy,i0:i0+nxy]

    dxy = np.round(hdr['CDELT2'],5)
    pix0 = int((nxy-1)/2)+int((nxy+1)/2)
    widxy = nxy*dxy
    
    if taper:      
        x, y = np.meshgrid(np.linspace(-(nxy-1)/2,(nxy-1)/2,nxy), 
                       np.linspace(-(nxy-1)/2,(nxy-1)/2,nxy))
        r = np.sqrt(x**2+y**2)
        
        taper = gf(nxy/2-taper_width,taper_width,r)
        taper[np.where(r<=nxy/2-taper_width)] = 1
        image_small = image_small*taper
        plt.plot(taper[511,:])
        
    image = np.zeros([2*nxy+1,2*nxy+1])
    image[:,:] = np.nanmean(image_small)
    image[int((nxy+1)/2):int((nxy+1)/2)+nxy,int((nxy+1)/2):int((nxy+1)/2)+nxy] = image_small

    print('Width of each pixel: '+str(dxy)+ ' deg.')
    print('Number of x and y pixels: '+str(nxy))
    print('Central pixel index: '+str(pix0))
    print('Width of image: '+str(widxy)+' deg.')
    
    return image,dxy,pix0,widxy

### ST and SA simulation functions

In [ ]:
def ST_observe(image,image_ST_true,dxy,pix0,
               R1=0,R2=13,plots=True,bounds=300):

    # Fourier transform full-scale image:
    image_FFT = np.fft.fft2(image)
    image_FFT_shift = np.fft.fftshift(image_FFT)

    # Fourier transform actual ST image:  
    image_FFT_true = np.fft.fft2(image_ST_true)
    image_FFT_true_shift = np.fft.fftshift(image_FFT_true)

    # Generate uv-plane axes and high-pass mask
    uv = get_uv_axis(image, dxy)
    idx, extent = get_uvbounds_idx(uv,bounds,bounds)
    mask = make_uv_mask('hp', R1, R2, uv['ruv'])

    # Apply the high-pass mask and Fourier transform back to image:
    image_FFT_shift_mask = image_FFT_shift*mask
    image_ST = np.fft.ifft2(np.fft.ifftshift(image_FFT_shift_mask))
    
    if plots:
    
        fig,ax = plt.subplots(1,1,figsize=(6,2))
        ax.plot(uv['ruv'][pix0,:],mask[pix0,:]),ax.set_xlim(0,50),ax.grid()
        
        # Plot full-scale image and FFT:
        fig,ax = plt.subplots(2,3,figsize=(16,10))
        ax[1,0].set_title('Original image')
        ax[1,0].imshow(image,origin='lower',vmin=0,vmax=30)
        ax[0,0].imshow(abs(image_FFT_shift)[idx['u1']:idx['u2']+1,idx['v1']:idx['v2']+1],
                       origin='lower',vmin=0,vmax=1e5,extent=extent)

        # Plot simulated ST image and FFT:
        ax[1,1].set_title('Simulated ST')
        ax[1,1].imshow(image_ST.real,origin='lower',vmin=-20,vmax=20,cmap='RdBu_r')
        ax[0,1].imshow(abs(image_FFT_shift_mask)[idx['u1']:idx['u2']+1,idx['v1']:idx['v2']+1],
                       origin='lower',vmin=0,vmax=1e5,extent=extent)

        # Plot actual ST image and FFT:
        ax[1,2].set_title('Actual ST')
        ax[1,2].imshow(image_ST_true,origin='lower',vmin=-0.15,vmax=0.15,cmap='RdBu_r')
        ax[0,2].imshow(abs(image_FFT_true_shift)[idx['u1']:idx['u2']+1,idx['v1']:idx['v2']+1],
                       origin='lower',vmin=0,vmax=1e3,extent=extent)
        

    print(np.nanmin(image_ST.real))
    print(np.nanmax(image_ST.real))
    return image_ST, mask

In [ ]:
def SA_observe(image,dxy,pix0,
               R=9,res1=1,res2=36,plots=True,bounds=300,method='fourier'):

    # Fourier transform full-scale image:
    image_FFT = np.fft.fft2(image)
    image_FFT_shift = np.fft.fftshift(image_FFT)

    # Generate uv-plane axes and Gaussian beam low-pass mask:
    uv = get_uv_axis(image, dxy)
    idx, extent = get_uvbounds_idx(uv,bounds,bounds)    
    mask = gf(0,2*R,uv['ruv'])

    if method == 'fourier':
        # Apply the FFT-based beam mask and Fourier transform back to image:
        image_FFT_shift_mask = image_FFT_shift*mask
        image_SA = np.fft.ifft2(np.fft.ifftshift(image_FFT_shift_mask))

    if method == 'convolve':
        # Apply convolution with Gaussian kernel:
        image_SA, kernel = convolve_to_beam(image, res2/60, res1/60, dxy, output = True)
        
    
    if plots:
        fig,ax = plt.subplots(2,2,figsize=(12,10))

        # Plot full-scale image:
        ax[1,0].set_title('Original image')
        ax[1,0].imshow(image,origin='lower',vmin=0,vmax=30)

        # Plot simulated SA image:
        ax[1,1].set_title('Beam mask applied')
        ax[1,1].imshow(image_SA.real,origin='lower',vmin=0,vmax=30)
          
        # Plot beam in Fourier or image plane:
        if method == 'fourier':
            ax[0,0].imshow(abs(image_FFT_shift)[idx['u1']:idx['u2']+1,idx['v1']:idx['v2']+1],
                           origin='lower',vmin=0,vmax=1e5,extent=extent)
            ax[0,1].imshow(abs(image_FFT_shift_mask)[idx['u1']:idx['u2']+1,idx['v1']:idx['v2']+1],
                           origin='lower',vmin=0,vmax=1e5,extent=extent)
        if method == 'convolve':
            ax[0,0].imshow(kernel,origin='lower')
            ax[0,1].imshow(kernel,origin='lower')

    
    return image_SA, mask


In [ ]:
def SA_deconvolve2(SA_image, SA_beam, noisecoeff, plots=True, *args,**kwargs):
    
    # FFT of input SA image:
    image_FFT = np.fft.fft2(SA_image)
    image_FFT_shift = np.fft.fftshift(image_FFT)

    # Get uv-plane axes:
    uv_freq = np.fft.fftshift(np.fft.fftfreq(SA_image.shape[0]))/dxy
    #uv_m = uv_freq*(3e8/1420e6)*180/(np.pi**2)
    uv_m = uv_freq*(3e8/(1420e6))*180/np.pi # corrected Oct 2024
    xuv, yuv = np.meshgrid(uv_m, uv_m)
    ruv = np.sqrt(xuv**2+yuv**2)
    extent=[uv_m[0],uv_m[-1],uv_m[0],uv_m[-1]]
    
    # Make noise array for FFT of image:
    noiseRe = noisecoeff*np.random.normal(size=([SA_image.shape[0],SA_image.shape[1]]))
    noiseIm = noisecoeff*np.random.normal(size=([SA_image.shape[0],SA_image.shape[1]]))

    # Make noisy version of SA image FFT:
    #image_FFT_shift_noisy = np.empty_like(image_FFT_shift)
    image_FFT_shift_noisy  = image_FFT_shift + (noiseRe+1j*noiseIm) 

    # Deconvolve with SA beam:
    SA_beam[SA_beam<1e-100] = 1e-100
    image_FFT_deconvolved  = image_FFT_shift_noisy/SA_beam
   
    if plots:
        lims = 50
        print(image_FFT_shift.shape)
        fig,ax = plt.subplots(1,1,figsize=(8,4))
        ax.scatter(uv_m,abs(image_FFT_shift[511,:]))
        ax.scatter(uv_m,abs(image_FFT_shift_noisy[511,:]))
        ax.scatter(uv_m,abs(image_FFT_deconvolved[511,:]))
        ax.plot(uv_m,abs(image_FFT_shift[511,:]))
        ax.plot(uv_m,abs(image_FFT_shift_noisy[511,:]))
        ax.plot(uv_m,abs(image_FFT_deconvolved[511,:]))
        
        ax.set_ylim(1e-3,1e8)
        ax.set_yscale('log')
        ax.set_xlim(-lims,lims)
        ax.grid()

        # Plot simulated observation (SA image):
        fig,ax = plt.subplots(2,3,figsize=(12,10))
        ax[1,0].set_title('Original SA image')
        ax[1,0].imshow(SA_image.real,origin='lower',vmin=0,vmax=30)
        ax[0,0].imshow(abs(image_FFT_shift),origin='lower',vmin=0,vmax=5e5,extent=extent)
        ax[0,0].set_xlim(-lims,lims)
        ax[0,0].set_ylim(-lims,lims)

        ax[1,1].set_title('Noisy SA image')
        ax[1,1].imshow(np.fft.ifft2(np.fft.ifftshift(image_FFT_shift_noisy)).real,origin='lower',vmin=0,vmax=30)
        ax[0,1].imshow(abs(image_FFT_shift_noisy),origin='lower',vmin=0,vmax=5e5,extent=extent)
        ax[0,1].set_xlim(-lims,lims)
        ax[0,1].set_ylim(-lims,lims)

        ax[1,2].set_title('deconvolved SA image')
        ax[1,2].imshow(np.fft.ifft2(np.fft.ifftshift(image_FFT_deconvolved)).real,origin='lower',vmin=0,vmax=30)
        ax[0,2].imshow(abs(image_FFT_deconvolved),origin='lower',vmin=0,vmax=5e5,extent=extent)
        ax[0,2].set_xlim(-lims,lims)
        ax[0,2].set_ylim(-lims,lims)
    
    #
    #image_FFT_shift[SA_beam>1e-300] = image_FFT_shift[SA_beam>1e-300]+noise[SA_beam>1e-300]/SA_beam[SA_beam>1e-300]
    #image_FFT_shift[SA_beam<1e-300] = 1e300
    #print('Threshold for zero: ',np.min(ruv[SA_beam<1e-300]))
    
    #if plots:
    #    # Plot deconvolved:
    #    ax[1,1].set_title('SA beam deconvolved')
    #    ax[0,1].imshow(abs(image_FFT_shift),origin='lower',vmin=0,vmax=1e5,extent=extent)
    #    ax[1,1].plot(uv_m,np.log10(abs(image_FFT_shift[511,:])))
    #    ax[1,1].plot(uv_m,np.log10(abs(image_full_FFT_shift[511,:])))
    #    ax[1,1].set_ylim(0,10),ax[1,1].set_xlim(-40,40),ax[1,1].grid()
     
    return image_FFT_deconvolved

### Feathering functions

In [ ]:
def feather_ST(image,dxy,pix0,R1=12.9,R2=17.1,*arg,**kwargs):
        
    image_FFT = np.fft.fft2(image)
    image_FFT_shift = np.fft.fftshift(image_FFT)

    #uv_freq = np.fft.fftshift(np.fft.fftfreq(image.shape[0]))/dxy
    #uv_m = uv_freq*(3e8/1420e6)*180/(np.pi**2)
    #uv_m = 2*uv_freq*(3e8/(1420e6))*180/np.pi # corrected Oct 2024
    #print(uv_m) # Units of 1/deg

    #xuv, yuv = np.meshgrid(uv_m, uv_m)
    #ruv = np.sqrt(xuv**2+yuv**2)

    uv = get_uv_axis(image, dxy)
    #idx, extent = get_uvbounds_idx(uv,bounds,bounds) 

    #extent=[uv_m[0],uv_m[-1],uv_m[0],uv_m[-1]]
    
    #b = np.pi/(R2-R1)
    #r0 = (R1+R2)/2.
    #mask = 0.5*np.sin(b*(ruv - r0))+0.5
    #mask[ruv > R2] = 1
    #mask[ruv < R1] = 0

    mask = make_uv_mask('hp', R1, R2, uv['ruv'])
    
    image_FFT_shift = image_FFT_shift*mask
    image_ST = np.fft.ifft2(np.fft.ifftshift(image_FFT_shift))
  
    return image_ST, mask, uv

In [ ]:
def feather_SA(image,FFT,dxy,pix0,R1=12.9,R2=17.1,*arg,**kwargs):
        
    #image_FFT = np.fft.fft2(image)
    #image_FFT_shift = np.fft.fftshift(image_FFT)

    #uv_freq = np.fft.fftshift(np.fft.fftfreq(image.shape[0]))/dxy
    #uv_m = uv_freq*(3e8/1420e6)*180/(np.pi**2)
    #uv_m = 2*uv_freq*(3e8/(1420e6))*180/np.pi # corrected Oct 2024
    #print(uv_m) # Units of 1/deg

    #xuv, yuv = np.meshgrid(uv_m, uv_m)
    #ruv = np.sqrt(xuv**2+yuv**2)

    uv = get_uv_axis(image, dxy)

    #extent=[uv_m[0],uv_m[-1],uv_m[0],uv_m[-1]]
    
    #b = np.pi/(R2-R1)
    #r0 = (R1+R2)/2.
    #mask = -0.5*np.sin(b*(ruv - r0))+0.5
    #mask[ruv > R2] = 0
    #mask[ruv < R1] = 1

    mask = make_uv_mask('lp', R1, R2, uv['ruv'])
    
    FFT = FFT*mask
    #plt.imshow(abs(FFT),vmin=0,vmax=1e5)
    
    image_SA = np.fft.ifft2(np.fft.ifftshift(FFT))
       
  
    return image_SA, mask, uv

In [ ]:
def simple_gap(image,dxy,pix0,R1=14,R2=18,R3=9,R4=13,*arg,**kwargs):
        
    image_FFT = np.fft.fft2(image)
    image_FFT_shift = np.fft.fftshift(image_FFT)

    uv_freq = np.fft.fftshift(np.fft.fftfreq(image.shape[0]))/dxy
    #uv_m = uv_freq*(3e8/1420e6)*180/(np.pi**2)
    uv_m = 2*uv_freq*(3e8/(1420e6))*180/np.pi # corrected Oct 2024
    #print(uv_m) # Units of 1/deg

    xuv, yuv = np.meshgrid(uv_m, uv_m)
    ruv = np.sqrt(xuv**2+yuv**2)

    #extent=[uv_m[0],uv_m[-1],uv_m[0],uv_m[-1]]
    
    b = np.pi/(R2-R1)
    r0 = (R1+R2)/2.
    mask1 = 0.5*np.sin(b*(ruv - r0))+0.5
    mask1[ruv > R2] = 1
    mask1[ruv < R1] = 0
    
    b = np.pi/(R4-R3)
    r0 = (R3+R4)/2.
    mask2 = -0.5*np.sin(b*(ruv - r0))+0.5
    mask2[ruv > R4] = 0
    mask2[ruv < R3] = 1
    
    mask = mask1+mask2
    
    image_FFT_shift = image_FFT_shift*mask
    image_new = np.fft.ifft2(np.fft.ifftshift(image_FFT_shift))
  
    return image_new, mask, uv_m

### Optimizing feathering parameters

In [ ]:
def feathering_tests(ST,image_full,SA_deconv_FFT,dxy,pix0,dR):

    R1_arr = np.arange(3,26,dR)
    R2_arr = np.arange(3,26,dR)

    mindiff = np.empty([len(R2_arr),len(R1_arr)])
    maxdiff = np.empty([len(R2_arr),len(R1_arr)])
    print(mindiff.shape)

    #for freq_idx in tqdm(range(0, len(freq))):
    for j in tqdm(range(0,len(R2_arr))):
        #print(j)
        for i in range(0,len(R1_arr)):
            R1 = R1_arr[i]
            R2 = R2_arr[j]
            if R2>R1:
                ST_feather,ST_mask,uv_m_ST = feather_ST(ST,dxy,pix0,R1=R1,R2=R2)
                SA_feather,SA_mask,uv_m_SA = feather_SA(image_full,SA_deconv_FFT,dxy,pix0,R1=R1,R2=R2)
                frac_diff = (abs(SA_feather+ST_feather)-image_full)/image_full
                mindiff[j,i] = np.min(frac_diff)*100
                maxdiff[j,i] = np.max(frac_diff)*100
            else:
                mindiff[j,i] = np.nan
                maxdiff[j,i] = np.nan
                
    return R1_arr, R2_arr, mindiff, maxdiff


def find_optimized_feathering(R1_arr,R2_arr,dR,mindiff,maxdiff,SA_noise,ST_R):

    w_low = np.where(maxdiff == np.nanmin(maxdiff))
    print(maxdiff[w_low][0])
    print(R1_arr[w_low[1]])
    print(R2_arr[w_low[0]])

    fs = 16
    fig,ax = plt.subplots(1,figsize=(10,10))
    extent = (R1_arr[0]-dR/2,R1_arr[-1]+dR/2,R2_arr[0]-dR/2,R2_arr[-1]+dR/2)
    im = ax.imshow(maxdiff,vmin=1,vmax=20,extent=extent,origin='lower')
    ax.set_xlabel('Maximum SA baseline, R1 (m)',fontsize = fs)
    ax.set_ylabel('Minimum ST baseline, R2 (m)',fontsize = fs)
    ax.set_title('ST edge = '+str(round(ST_R,1))+' m,   SA noise slope = '+str(round(SA_noise,1)),fontsize=fs)
    #ax.grid()

    ax.scatter(R1_arr[w_low[1]],R2_arr[w_low[0]],s=5,color='r')
    ax.text(14,10,'R1 = '+str(round(R1_arr[w_low[1][0]],1))+' m',fontsize = fs)
    ax.text(14,8,'R2 = '+str(round(R2_arr[w_low[0][0]],1))+' m',fontsize = fs)
    ax.text(14,6,'Max diff. = '+str(round(maxdiff[w_low][0],1))+' %',fontsize = fs)
    ax.tick_params(axis='both', labelsize=fs)

    divider = make_axes_locatable(ax)
    cax = divider.append_axes('right', size='5%', pad=0.05)
    cbar = fig.colorbar(im, cax=cax, orientation='vertical')
    cbar.ax.tick_params(labelsize=fs) 
    cbar.set_label('Percent difference', fontsize=fs)

    plt.tight_layout()
    plt.savefig('/home/ordoga/Python/CGPS_GMIMS_PLOTS/optimize_uv_'+str(int(ST_R))+'_'+str(int(SA_noise))+'.png')
    
    return

## Make images from full-uv-coverage data and ST-only data (for comparison)

In [ ]:
image_full,dxy,pix0,widxy = regular_image(data_full,400,22100,1023)
print('')
image_ST_true,dxy,pix0,widxy = regular_image(data_ST,400,22555,1023)

fig,ax = plt.subplots(1,2,figsize=(12,6)) 
ax[0].imshow(image_full,origin='lower',vmin=0,vmax=30), ax[0].set_title('Full ST+SA image')
ax[1].imshow(image_ST_true,origin='lower',vmin=0,vmax=0.2),ax[1].set_title('ST only (actual)')

In [ ]:
print(image_full.shape)
print(image_ST_true.shape)

fig,axs = plt.subplots(2,1,figsize=(18,6))

axs[0].hist(image_full.flatten(),bins=101,alpha=0.5,range=(-100,100));
axs[1].hist(image_ST_true.flatten(),bins=101,alpha=0.5,range=(-0.1,0.1));

# (1) Single simulation with plots

## Simulate ST data from full-coverage data

In [ ]:
ST_map, ST_mask = ST_observe(image_full,image_ST_true,dxy,pix0,
                             R1=8.6,R2=12.9,plots=True)

#plt.savefig('../plots/uv_tests1_new_v2.png')

In [ ]:
fig,ax = plt.subplots(1,1,figsize=(8,8))
ax.scatter(image_ST_true.flatten(),ST_map.real.flatten(),s=0.1)
ax.set_xlim(-0.15,0.6)
ax.set_ylim(-15,60)
ax.grid()

## Simulate SA data from full-coverage data

In [ ]:
### Use convolution to simulate SA observation:
#SA_map, SA_mask = SA_observe(image_full,dxy,pix0,res1=1,res2=36,plots=True,method='convolve')

#plt.savefig('../plots/SA_test_conv.png')

In [ ]:
### Use Fourier filtering to simulate SA observation:
SA_map2, SA_mask2 = SA_observe(image_full,dxy,pix0,R=9,plots=True,method='fourier',bounds=50)

plt.savefig('../plots/SA_test_fourier9_nofactor2.png')

In [ ]:
## Check that Fourier method and convolving method give same result
#plt.imshow(SA_map.real-SA_map2.real,vmin=-3,vmax=3,cmap='RdBu_r')

## Deconvolve SA beam

In [ ]:
#SA_deconv_FFT = SA_deconvolve(SA, image_full, SA_beam)
SA_noise = 1000
SA_deconv_FFT = SA_deconvolve2(SA_map2, SA_mask2, SA_noise,plots=True)

In [ ]:
print(uv['ruv'])

## Feather simulated data in uv-plane and combine in image plane

In [ ]:
#ST_feather,ST_mask,uv_m_ST = feather_ST(ST,dxy,pix0,R1=9,R2=17)
#SA_feather,SA_mask,uv_m_SA = feather_SA(image_full,SA_deconv_FFT,dxy,pix0,R1=9,R2=17)

ST_feather,ST_feather_mask,uv_ST = feather_ST(ST_map,dxy,pix0,R1=8.6,R2=17.1)
SA_feather,SA_feather_mask,uv_SA = feather_SA(image_full,SA_deconv_FFT,dxy,pix0,R1=8.6,R2=17.1)

fig,ax = plt.subplots(1,1,figsize=(6,3))

#print(uv_ST)

#ax.fill_between(X, Y, 0, color='blue', alpha=.1)

print(uv_ST.keys())
print(uv_ST['ruv'].shape)
print(pix0)

ax.plot(uv_ST['uvm'],ST_feather_mask[pix0,:],color='red')
ax.plot(uv_SA['uvm'],SA_feather_mask[pix0,:],color='blue')
ax.plot(uv_ST['uvm'],ST_mask[pix0,:],color='k',linewidth=0.5)
ax.plot(uv_SA['uvm'],SA_mask2[pix0,:],color='k',linewidth=0.5)
ax.axvline(x=8.6,color='blue',linestyle='dashed')
ax.axvline(x=17.1,color='red',linestyle='dashed')
##ax.plot(r_SA[511:1024],mask_SA[511:1024]+mask_ST[511:1024],color='k')
ax.fill_between(uv_ST['uvm'],ST_mask[pix0,:],color='gray',alpha=0.2,linewidth=2)
ax.fill_between(uv_SA['uvm'],SA_mask2[pix0,:],color='gray',alpha=0.2,linewidth=2)
ax.set_xlim(0,30)
ax.set_ylim(0,1.1)
ax.set_xlabel('Baseline (m)')
plt.tight_layout()
#plt.savefig('../CGPS_GMIMS_plots/filter_sketch.png')

#print(r_ST[511:1024])

#fig,ax = plt.subplots(1,3,figsize=(20,6))
#ax[0].imshow(SA_feather.real,origin='lower',vmin=0,vmax=30)
#ax[1].imshow(ST_feather.real,origin='lower',vmin=0,vmax=30)
#ax[2].imshow(abs(SA_feather+ST_feather),origin='lower',vmin=0,vmax=30)


In [ ]:
fig,ax = plt.subplots(1,3,figsize=(20,6))
ax[0].imshow(abs(SA_feather+ST_feather),origin='lower',vmin=0,vmax=30)
ax[1].imshow(image_full,origin='lower',vmin=0,vmax=30)

#frac_diff = (abs(SA_feather+ST_feather)-image_full)/image_full
frac_diff = abs(SA_feather+ST_feather)-image_full

ax[2].imshow(frac_diff,origin='lower',vmin=-3,vmax=3,cmap='RdBu_r')
print(np.min(frac_diff),np.max(frac_diff))

In [ ]:
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable

In [ ]:
fig,ax = plt.subplots(1,5,figsize=(17,4))
fs = 16

im = ax[0].imshow(image_full,origin='lower',vmin=0,vmax=30,cmap='gray')
ax[0].set_title('Full-scale (true) image',fontsize=fs)
ax2_divider = make_axes_locatable(ax[0])
cax2 = ax2_divider.append_axes("bottom", size="5%", pad="2%")
cb2 = fig.colorbar(im, cax=cax2, orientation="horizontal",label='K')

im = ax[1].imshow(SA_map,origin='lower',vmin=0,vmax=30,cmap='gray')
ax[1].set_title('Single dish observation',fontsize=fs)
ax2_divider = make_axes_locatable(ax[1])
cax2 = ax2_divider.append_axes("bottom", size="5%", pad="2%")
cb2 = fig.colorbar(im, cax=cax2, orientation="horizontal",label='K')

im = ax[2].imshow(ST_map,origin='lower',vmin=0,vmax=10,cmap='gray')
ax[2].set_title('Interferometer observation',fontsize=fs)
ax2_divider = make_axes_locatable(ax[2])
cax2 = ax2_divider.append_axes("bottom", size="5%", pad="2%")
cb2 = fig.colorbar(im, cax=cax2, orientation="horizontal",label='K')

im = ax[3].imshow(abs(SA_feather+ST_feather),origin='lower',vmin=0,vmax=30,cmap='gray')
ax[3].set_title('Combined reconstruction',fontsize=fs)
ax2_divider = make_axes_locatable(ax[3])
cax2 = ax2_divider.append_axes("bottom", size="5%", pad="2%")
cb2 = fig.colorbar(im, cax=cax2, orientation="horizontal",label='K')

frac_diff = (abs(SA_feather+ST_feather)-image_full)/image_full
im = ax[4].imshow(frac_diff,origin='lower',vmin=-0.05,vmax=0.05,cmap='bwr')
ax[4].set_title('True - combined',fontsize=fs)
ax2_divider = make_axes_locatable(ax[4])
cax2 = ax2_divider.append_axes("bottom", size="5%", pad="2%")
cb2 = fig.colorbar(im, cax=cax2, orientation="horizontal",label='Fractional residuals')

print(np.min(frac_diff),np.max(frac_diff))

for i in range(0,5):
    ax[i].xaxis.set_tick_params(labelbottom=False)
    ax[i].yaxis.set_tick_params(labelleft=False)
    ax[i].set_xticks([])
    ax[i].set_yticks([])

#plt.tight_layout()
#plt.savefig('../CGPS_GMIMS_plots/filtering.png')

In [ ]:
fig,ax = plt.subplots(2,3,figsize=(20,12))
fs = 20

im = ax[0,0].imshow(image_full,origin='lower',vmin=0,vmax=30,cmap='gray')
ax[0,0].set_title('(a) Full-scale (true) image',fontsize=fs+4)
ax2_divider = make_axes_locatable(ax[0,0])
cax2 = ax2_divider.append_axes("right", size="5%", pad="2%")
cb2 = fig.colorbar(im, cax=cax2, orientation="vertical")
cb2.ax.tick_params(labelsize=fs) 
cb2.set_label('K', fontsize=fs)

im = ax[0,1].imshow(SA_map2.real,origin='lower',vmin=0,vmax=30,cmap='gray')
ax[0,1].set_title('(b) SA observation',fontsize=fs+4)
ax2_divider = make_axes_locatable(ax[0,1])
cax2 = ax2_divider.append_axes("right", size="5%", pad="2%")
cb2 = fig.colorbar(im, cax=cax2, orientation="vertical")
cb2.ax.tick_params(labelsize=fs) 
cb2.set_label('K', fontsize=fs)

im = ax[0,2].imshow(ST_map.real,origin='lower',vmin=0,vmax=10,cmap='gray')
ax[0,2].set_title('(c) AS observation',fontsize=fs+4)
ax2_divider = make_axes_locatable(ax[0,2])
cax2 = ax2_divider.append_axes("right", size="5%", pad="2%")
cb2 = fig.colorbar(im, cax=cax2, orientation="vertical")
cb2.ax.tick_params(labelsize=fs) 
cb2.set_label('K', fontsize=fs)


ax[1,0].plot(uv_ST['uvm'],ST_feather_mask[pix0,:],color='red',linewidth=2)
ax[1,0].plot(uv_SA['uvm'],SA_feather_mask[pix0,:],color='blue',linewidth=2)
ax[1,0].plot(uv_ST['uvm'],ST_mask[pix0,:],color='k',linewidth=1)
ax[1,0].plot(uv_SA['uvm'],SA_mask2[pix0,:],color='k',linewidth=1)
#ax.axvline(x=8.6,color='blue',linestyle='dashed')
#ax.axvline(x=12.86,color='red',linestyle='dashed')
##ax.plot(r_SA[511:1024],mask_SA[511:1024]+mask_ST[511:1024],color='k')
ax[1,0].fill_between(uv_ST['uvm'],ST_mask[pix0,:],color='gray',alpha=0.2,linewidth=2)
ax[1,0].fill_between(uv_SA['uvm'],SA_mask2[pix0,:],color='gray',alpha=0.2,linewidth=2)
ax[1,0].set_xlim(0,30)
ax[1,0].set_ylim(0,1.1)
ax[1,0].set_xlabel('Baseline (m)',fontsize=fs)
ax[1,0].set_title('(d) $uv$ coverage and filtering',fontsize=fs+4)
ax[1,0].tick_params(labelsize=fs)

im = ax[1,1].imshow(abs(SA_feather+ST_feather),origin='lower',vmin=0,vmax=30,cmap='gray')
ax[1,1].set_title('(e) Combined reconstruction',fontsize=fs+4)
ax2_divider = make_axes_locatable(ax[1,1])
cax2 = ax2_divider.append_axes("right", size="5%", pad="2%")
cb2 = fig.colorbar(im, cax=cax2, orientation="vertical")
cb2.ax.tick_params(labelsize=fs) 
cb2.set_label('K', fontsize=fs)

frac_diff = (abs(SA_feather+ST_feather)-image_full)/image_full
im = ax[1,2].imshow(frac_diff,origin='lower',vmin=-0.05,vmax=0.05,cmap='bwr')
ax[1,2].set_title('(f) (True - combined)/true',fontsize=fs+4)
ax2_divider = make_axes_locatable(ax[1,2])
cax2 = ax2_divider.append_axes("right", size="5%", pad="2%")
cb2 = fig.colorbar(im, cax=cax2, orientation="vertical")
cb2.ax.tick_params(labelsize=fs) 
cb2.set_label('Fractional residuals', fontsize=fs)

print(np.min(frac_diff),np.max(frac_diff))

for i in range(0,3):
    for j in range(0,2):
        if j == 0:
            ax[j,i].xaxis.set_tick_params(labelbottom=False)
            ax[j,i].yaxis.set_tick_params(labelleft=False)
            ax[j,i].set_xticks([])
            ax[j,i].set_yticks([])
        else:
            if i != 0:
                ax[j,i].xaxis.set_tick_params(labelbottom=False)
                ax[j,i].yaxis.set_tick_params(labelleft=False)
                ax[j,i].set_xticks([])
                ax[j,i].set_yticks([])

plt.tight_layout()
plt.savefig('../plots/uv_simulation.pdf')

In [ ]:
plt.hist(frac_diff.flatten(),bins=101,range=(-0.1,0.1));
print(len(np.where(frac_diff.flatten()<0.02)[0]))
print(len(frac_diff.flatten()))
print(100*len(np.where(frac_diff.flatten()<0.02)[0])/len(frac_diff.flatten()))

In [ ]:
0.005*1024

## Try different combinations of feathering parameters and optimize:

In [ ]:
dR = 0.5
R1_arr, R2_arr, mindiff, maxdiff = feathering_tests(ST,image_full,SA_deconv_FFT,dxy,pix0,dR)
find_optimized_feathering(R1_arr,R2_arr,dR,mindiff,maxdiff,SA_noise,ST_R)


# (2) Run different simulations and optimize feathering for each

In [ ]:
SA_noise_arr = [0, 500, 1000, 1500, 2000, 2500, 3000, 3500, 4000]
ST_R_arr = [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]

dR = 0.5

for SA_noise in SA_noise_arr:
    for ST_R in ST_R_arr:
        
        print('--------------------------------------------------')
        print('Simulation for SA noise = '+str(SA_noise)+' and ST edge = '+str(ST_R))
        print('--------------------------------------------------')
        ST = ST_observe(image_full,image_ST_true,dxy,pix0,R2=ST_R,plots=False)
        SA_beam = get_beam(image_full,dxy,pix0,R=9,plots=False)
        SA_deconv_FFT = SA_deconvolve2(image_full, SA_beam, SA_noise,plots=False)

        R1_arr, R2_arr, mindiff, maxdiff = feathering_tests(ST,image_full,SA_deconv_FFT,dxy,pix0,dR)
        find_optimized_feathering(R1_arr,R2_arr,dR,mindiff,maxdiff,SA_noise,ST_R)
        
        print('')

In [ ]:
gap_image,gap_mask,uv_m = simple_gap(image_full,dxy,pix0)
plt.plot(uv_m,gap_mask[pix0,:])
plt.xlim(0,50)
plt.grid()

fig,ax = plt.subplots(1,3,figsize=(20,6))
ax[0].imshow(abs(gap_image),origin='lower',vmin=0,vmax=30)
ax[1].imshow(image_full,origin='lower',vmin=0,vmax=30)

frac_diff = (abs(gap_image)-image_full)/image_full

ax[2].imshow(frac_diff,origin='lower',vmin=-0.05,vmax=0.05)
print(np.min(frac_diff),np.max(frac_diff))

### Old code

In [ ]:
### NOTE: OBSOLETE!!!
def SA_observe_old(image,dxy,pix0,noise_slope=0,R=9,Rnoise=9,*arg,**kwargs):
    
    # Note: default is no noise added (noise_slope = 0)
    
    image_FFT = np.fft.fft2(image)
    image_FFT_shift = np.fft.fftshift(image_FFT)

    uv_freq = np.fft.fftshift(np.fft.fftfreq(image.shape[0]))/dxy
    uv_m = uv_freq*(3e8/1420e6)*180/(np.pi**2)

    xuv, yuv = np.meshgrid(uv_m, uv_m)
    ruv = np.sqrt(xuv**2+yuv**2)
    
    extent=[uv_m[0],uv_m[-1],uv_m[0],uv_m[-1]]
    
    mask = gf(0,2*R,ruv)
    noise = np.random.normal(size=([image.shape[0],image.shape[1]]))
    noise = noise*np.sqrt(ruv-Rnoise)*noise_slope
    noise[ruv<Rnoise] = 0

    fig,ax = plt.subplots(2,2,figsize=(12,10))
    ax[1,0].set_title('Beam mask')
    ax[0,0].imshow(mask,origin='lower',vmin=0,vmax=1,extent=extent)
    ax[1,0].plot(uv_m,mask[pix0,:]), ax[1,0].set_box_aspect(1)
    ax[1,1].set_title('Added noise')
    ax[0,1].imshow(noise,origin='lower',vmin=-200,vmax=200,extent=extent)
    ax[1,1].plot(uv_m,noise[pix0,:]), ax[1,1].set_box_aspect(1)
    
    # Plot original image:
    fig,ax = plt.subplots(2,3,figsize=(16,10))
    ax[1,0].set_title('Original image')
    ax[1,0].imshow(image,origin='lower',vmin=0,vmax=30)
    ax[0,0].imshow(abs(image_FFT_shift),origin='lower',vmin=0,vmax=1e5,extent=extent)
    
    # Apply beam mask and plot:
    image_FFT_shift = image_FFT_shift*mask
    image_SA_nonoise = np.fft.ifft2(np.fft.ifftshift(image_FFT_shift))
    ax[1,1].set_title('Beam mask applied')
    ax[1,1].imshow(image_SA_nonoise.real,origin='lower',vmin=0,vmax=30)
    ax[0,1].imshow(abs(image_FFT_shift),origin='lower',vmin=0,vmax=1e5,extent=extent)
        
    # Apply noise and plot:    
    image_FFT_shift = image_FFT_shift+noise
    image_SA = np.fft.ifft2(np.fft.ifftshift(image_FFT_shift))
    ax[1,2].set_title('Simulated SA obs: beam + noise applied')
    ax[1,2].imshow(image_SA.real,origin='lower',vmin=0,vmax=30)
    ax[0,2].imshow(abs(image_FFT_shift),origin='lower',vmin=0,vmax=1e5,extent=extent)

    #mask = np.zeros_like(image)
    #npix1 = 2*int(np.round(R*np.diff(uv_m)[0],0))
    #npix = npix1*2+1
    #window1d = np.hanning(npix)
    #window2d = np.outer(window1d,window1d)
    #mask[pix0-npix1:pix0+npix1+1,pix0-npix1:pix0+npix1+1] = window2d    
    
    return image_SA, mask, noise

In [ ]:
### NOTE: OBSOLETE!!!
def SA_deconvolve_old(image, image_full, SA_beam):
    
    image_FFT = np.fft.fft2(image)
    image_FFT_shift = np.fft.fftshift(image_FFT)
    
    image_full_FFT = np.fft.fft2(image_full)
    image_full_FFT_shift = np.fft.fftshift(image_full_FFT)

    uv_freq = np.fft.fftshift(np.fft.fftfreq(image.shape[0]))/dxy
    #uv_m = uv_freq*(3e8/1420e6)*180/(np.pi**2)
    uv_m = 2*uv_freq*(3e8/(1420e6))*180/np.pi # corrected Oct 2024

    xuv, yuv = np.meshgrid(uv_m, uv_m)
    ruv = np.sqrt(xuv**2+yuv**2)
    
    extent=[uv_m[0],uv_m[-1],uv_m[0],uv_m[-1]]
    
    # Plot simulated observation (SA image):
    fig,ax = plt.subplots(2,2,figsize=(12,10))
    ax[1,0].set_title('Simulated SA observation')
    ax[1,0].imshow(SA.real,origin='lower',vmin=0,vmax=30)
    ax[0,0].imshow(abs(image_FFT_shift),origin='lower',vmin=0,vmax=1e5,extent=extent)
    
    image_FFT_shift[SA_beam>1e-300] = image_FFT_shift[SA_beam>1e-300]/SA_beam[SA_beam>1e-300]
    image_FFT_shift[SA_beam<1e-300] = 1e300
    print('Threshold for zero: ',np.min(ruv[SA_beam<1e-300]))
    #SA_deconv = np.fft.ifft2(np.fft.ifftshift(image_FFT_shift))
            
    # Plot deconvolved:
    ax[1,1].set_title('SA beam deconvolved')
    ax[0,1].imshow(abs(image_FFT_shift),origin='lower',vmin=0,vmax=1e5,extent=extent)
    ax[1,1].plot(uv_m,np.log10(abs(image_FFT_shift[511,:])))
    ax[1,1].plot(uv_m,np.log10(abs(image_full_FFT_shift[511,:])))
    ax[1,1].set_ylim(2,8),ax[1,1].set_xlim(-40,40),ax[1,1].grid()
     
    return image_FFT_shift